# 轮动策略教程

本教程介绍如何用 open-xquant 构建一个**动量轮动策略**——在多个资产中，按风险调整后的动量排名，定期选择表现最优的资产并分配权重。

我们将使用三只跨资产类别 ETF 作为投资标的：

| 代码 | 名称 | 资产类别 |
|------|------|----------|
| 513100.SS | 纳指100ETF | 美股科技 |
| 510300.SS | 沪深300ETF | A股大盘 |
| 518880.SS | 黄金ETF | 贵金属 |

### 与均线策略的区别

在 `engine_module` 教程中，我们学习了 SMA 均线交叉策略——**单标的、固定仓位、信号驱动**。轮动策略则是一种完全不同的范式：

| 维度 | 均线交叉策略 | 轮动策略 |
|------|-------------|----------|
| 标的数 | 单标的 | **多标的截面选择** |
| 信号含义 | 买入/不买入（布尔） | **目标权重**（0.0~1.0） |
| 仓位管理 | 固定股数 or 全仓 | **按权重动态调仓** |
| Rule 类型 | EntryRule + ExitRule | **RebalanceRule** |

### 新组件一览

本教程将用到三个新组件：

- **Momentum** — N 日动量指标（平均日对数收益率）
- **RollingVolatility** — N 日滚动波动率
- **Ratio** — 两列之比（用于计算风险调整动量）
- **TopNRanking** — 截面排名信号，输出目标权重
- **RebalanceRule** — 按目标权重定期调仓

## 1. 准备数据

In [ ]:
from oxq.data import YFinanceDownloader

dl = YFinanceDownloader()
for s in ["513100.SS", "510300.SS", "518880.SS"]:
    path = dl.download(s, "2024-11-15", "2026-02-28")
    print(f"{s}: {path}")

In [ ]:
from oxq.data import LocalMarketDataProvider

market = LocalMarketDataProvider()
for s in ["513100.SS", "510300.SS", "518880.SS"]:
    df = market.get_bars(s, "2024-11-15", "2026-02-28")
    print(f"{s}: {len(df)} bars, {df.index[0].date()} ~ {df.index[-1].date()}")

---
## 2. Indicator 层 — Momentum, RollingVolatility, Ratio

轮动策略的核心思路：**买动量强、波动低的资产**。我们需要三个指标：

1. **Momentum(20)** — 过去 20 天的平均日对数收益率，衡量趋势强度
2. **RollingVolatility(20)** — 过去 20 天的滚动波动率，衡量风险
3. **Ratio(mom/vol)** — 动量除以波动率 = 风险调整动量（RAM）

RAM 越高，说明该资产趋势越强且波动越低——是轮动策略最想持有的。

In [ ]:
from oxq.indicators import Momentum, RollingVolatility, Ratio

# 以纳指100ETF为例，演示三个指标的计算过程
df = market.get_bars("513100.SS", "2024-11-15", "2026-02-28").copy()

df["mom"] = Momentum().compute(df, column="close", period=20)
df["vol"] = RollingVolatility().compute(df, column="close", period=20)
df["ram"] = Ratio().compute(df, col_a="mom", col_b="vol")

print("纳指100ETF 指标计算结果（最后 10 行）：")
print(df[["close", "mom", "vol", "ram"]].tail(10).to_string(float_format="%.4f"))

可以看到：
- `mom` — 前 20 行为 NaN（窗口不足），之后反映近 20 日的平均日收益率
- `vol` — 前 ~21 行为 NaN（log_returns.diff() 再 rolling），反映近 20 日的波动水平
- `ram` — mom/vol，值越大说明"趋势强且稳定"，是排名的依据

**这就是为什么策略开头有一段空仓期** — 指标需要约 21 个交易日的预热数据。

我们同时计算三只 ETF 的 RAM 并对比：

In [ ]:
import pandas as pd

SYMBOLS = {"513100.SS": "纳指100ETF", "510300.SS": "沪深300ETF", "518880.SS": "黄金ETF"}

ram_df = pd.DataFrame()
for sym, name in SYMBOLS.items():
    d = market.get_bars(sym, "2024-11-15", "2026-02-28").copy()
    d["mom"] = Momentum().compute(d, column="close", period=20)
    d["vol"] = RollingVolatility().compute(d, column="close", period=20)
    d["ram"] = Ratio().compute(d, col_a="mom", col_b="vol")
    ram_df[name] = d["ram"]

# 显示 2025 年 1 月的 RAM 截面对比
print("2025年1月 风险调整动量（RAM）截面对比：")
jan = ram_df.loc["2025-01-01":"2025-01-31"]
print(jan.to_string(float_format="%.4f"))

每天三只 ETF 都有各自的 RAM 值。**哪只 RAM 最高，就应该分配更多权重**——这正是下一步 TopNRanking 信号要做的事。

---
## 3. Signal 层 — TopNRanking

TopNRanking 是一个**截面排名信号**，与 Crossover 的区别：

| 维度 | Crossover | TopNRanking |
|------|-----------|-------------|
| 输出类型 | 布尔值（True/False） | **权重**（0.0~1.0） |
| 视角 | 单 symbol 时序 | **多 symbol 截面** |
| 含义 | "此刻是否应该买入" | "此刻应该分配多少仓位" |

TopNRanking 的处理流程（每个 bar）：

```
1. 读取每个 symbol 的评分列（如 ram）
2. 过滤 NaN 和负值（动量为负 = 下跌趋势，不选）
3. 按评分降序排名，取 Top N
4. 归一化为权重（评分越高，权重越大，总和 = 1.0）
5. 权重上限裁剪（如单只不超过 60%，超出部分归现金）
```

In [ ]:
from oxq.signals import TopNRanking

# 准备 mktdata: dict[symbol, DataFrame]（每个 DataFrame 包含 ram 列）
mktdata = {}
for sym in SYMBOLS:
    d = market.get_bars(sym, "2024-11-15", "2026-02-28").copy()
    d["mom"] = Momentum().compute(d, column="close", period=20)
    d["vol"] = RollingVolatility().compute(d, column="close", period=20)
    d["ram"] = Ratio().compute(d, col_a="mom", col_b="vol")
    mktdata[sym] = d

# TopNRanking: 选 Top 2，单只上限 60%
ranking = TopNRanking()
weights = ranking.compute(mktdata, score="ram", n=2, max_weight=0.6)

# 显示权重
weight_df = pd.DataFrame({SYMBOLS[s]: weights[s] for s in SYMBOLS})
print("2025年1月 目标权重：")
jan_w = weight_df.loc["2025-01-01":"2025-01-31"]
print(jan_w.to_string(float_format="%.4f"))

观察权重分配：
- 每天只有 **最多 2 只** ETF 获得非零权重（n=2）
- 单只权重不超过 **0.6**（max_weight=0.6），超出部分不会分给其他标的，而是留作现金
- 动量为负的标的权重为 0（被 filter_negative 过滤）
- 前 ~21 天全部为 0（指标预热期，RAM 为 NaN）

### 权重上限的设计意图

为什么超过 max_weight 的部分归现金而不是重新分配？

假设某天 A 的 RAM 远高于 B，归一化后 A=0.9, B=0.1。如果 max_weight=0.6：

- **归现金**（本策略采用）：A=0.6, B=0.1, Cash=0.3 — 自然降低了总仓位
- **重新分配**：A=0.6, B=0.4 — 人为放大了弱势标的的权重

前者更保守，在信号集中度高的时候自动降杠杆。

---
## 4. Rule 层 — RebalanceRule

RebalanceRule 与 EntryRule/ExitRule 的根本区别：

| 维度 | EntryRule / ExitRule | RebalanceRule |
|------|---------------------|---------------|
| 触发条件 | 布尔信号 | **周期性**（每 N 个 bar） |
| 订单方向 | 单向（只买 or 只卖） | **双向**（根据差异买或卖） |
| 仓位逻辑 | 固定股数 / 全仓 | **按目标权重计算** |

RebalanceRule 的工作流程：

```
1. 频率门控：bar_count % frequency != 0 → 跳过
2. 读取目标权重 → 计算目标股数 = portfolio_value × weight / price
3. 与当前持仓比较 → delta = 目标 - 当前
4. delta > 0 → BUY   |   delta < 0 → SELL   |   delta == 0 → None
```

In [ ]:
from oxq.core import Portfolio, Position
from oxq.rules import RebalanceRule

rule = RebalanceRule(weight_col="tw", frequency=1)  # frequency=1 方便演示

# 场景 1: 空仓 → 目标权重 50% → 买入
portfolio = Portfolio(cash=100_000.0)
row = pd.Series({"close": 1.5, "tw": 0.5}, name=pd.Timestamp("2025-01-06"))

order = rule.evaluate("513100.SS", row, portfolio)
print(f"场景1 — 空仓买入: {order}")
print(f"  目标股数 = int(100000 × 0.5 / 1.5) = {int(100_000 * 0.5 / 1.5)}")
print()

In [ ]:
# 场景 2: 超配 → 目标权重 30% → 卖出
rule2 = RebalanceRule(weight_col="tw", frequency=1)
portfolio2 = Portfolio(
    cash=50_000.0,
    positions={"513100.SS": Position(symbol="513100.SS", shares=30000, avg_cost=1.5)},
)
# total_value = 50000 + 30000*1.6 = 98000
row2 = pd.Series({"close": 1.6, "tw": 0.3}, name=pd.Timestamp("2025-02-01"))

order2 = rule2.evaluate("513100.SS", row2, portfolio2)
print(f"场景2 — 超配卖出: {order2}")
print(f"  总资产 = 50000 + 30000×1.6 = {50000 + 30000*1.6:.0f}")
print(f"  目标股数 = int(98000 × 0.3 / 1.6) = {int(98000 * 0.3 / 1.6)}")
print(f"  卖出 = 30000 - 18375 = {30000 - int(98000 * 0.3 / 1.6)}")
print()

In [ ]:
# 场景 3: 权重为 0 → 全部清仓
rule3 = RebalanceRule(weight_col="tw", frequency=1)
portfolio3 = Portfolio(
    cash=10_000.0,
    positions={"518880.SS": Position(symbol="518880.SS", shares=5000, avg_cost=7.0)},
)
row3 = pd.Series({"close": 7.5, "tw": 0.0}, name=pd.Timestamp("2025-03-01"))

order3 = rule3.evaluate("518880.SS", row3, portfolio3)
print(f"场景3 — 权重归零清仓: {order3}")

### 频率门控

实际使用中不会每天调仓（交易成本太高），而是每隔 N 天才触发一次：

In [ ]:
rule_f5 = RebalanceRule(weight_col="tw", frequency=5)
portfolio_f = Portfolio(cash=100_000.0)

dates = pd.bdate_range("2025-01-01", periods=10)
for i, date in enumerate(dates):
    row = pd.Series({"close": 1.5, "tw": 0.5}, name=date)
    order = rule_f5.evaluate("513100.SS", row, portfolio_f)
    status = f"→ {order.side} {order.shares}" if order else "→ 跳过"
    print(f"  Bar {i+1:2d} ({date.date()}) {status}")

只有第 5 和第 10 个 bar 触发了调仓（`bar_count % 5 == 0`），其余跳过。

---
## 5. 组装完整策略

现在把所有组件组合为一个完整的 Strategy：

```
Universe    → 513100.SS, 510300.SS, 518880.SS
  ↓
Indicator   → mom = Momentum(20)
            → vol = RollingVolatility(20)
            → ram = Ratio(mom / vol)
  ↓
Signal      → tw = TopNRanking(score=ram, n=2, max_weight=0.6)
  ↓
Rule        → RebalanceRule(weight_col=tw, frequency=10)
```

注意：轮动策略**不需要** EntryRule 和 ExitRule — RebalanceRule 同时负责买入和卖出。

In [ ]:
from oxq.core import Engine, Strategy
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse

strategy = Strategy(
    name="momentum_rotation",
    hypothesis=(
        "在纳指100ETF、沪深300ETF、黄金ETF中，"
        "按 Momentum(20)/Volatility(20) 风险调整动量排名，"
        "选 Top 2 归一化权重，单只上限 60%，"
        "每 10 天调仓可获得正超额收益"
    ),
    universe=StaticUniverse(("513100.SS", "510300.SS", "518880.SS")),
    indicators={
        "mom": (Momentum(), {"column": "close", "period": 20}),
        "vol": (RollingVolatility(), {"column": "close", "period": 20}),
        "ram": (Ratio(), {"col_a": "mom", "col_b": "vol"}),
    },
    signals={
        "tw": (TopNRanking(), {"score": "ram", "n": 2, "max_weight": 0.6}),
    },
    entry_rules=[],
    exit_rules=[],
    rebalance_rules=[RebalanceRule(weight_col="tw", frequency=10)],
)

print(f"策略名称: {strategy.name}")
print(f"指标: {list(strategy.indicators.keys())}")
print(f"信号: {list(strategy.signals.keys())}")
print(f"调仓规则: {len(strategy.rebalance_rules)} 条")
print(f"入场/出场规则: {len(strategy.entry_rules)} + {len(strategy.exit_rules)} 条（轮动策略不需要）")

---
## 6. 运行回测

In [ ]:
broker = SimBroker()
result = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    router=broker,
    receiver=broker,
    start="2024-11-15",
    end="2026-02-28",
    initial_cash=100_000.0,
)

print(f"总收益率:     {result.total_return():.2%}")
print(f"年化收益率:   {result.annualized_return():.2%}")
print(f"年化波动率:   {result.annualized_volatility():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"Calmar Ratio: {result.calmar_ratio():.2f}")
print(f"Sortino Ratio: {result.sortino_ratio():.2f}")
print(f"最大回撤:     {result.max_drawdown():.2%}")
print(f"交易次数:     {len(result.trades)}")
print(f"期末总资产:   {result.equity_curve[-1][1]:,.0f}")

---
## 7. 查看交易记录

In [ ]:
NAMES = {"513100.SS": "纳指100ETF", "510300.SS": "沪深300ETF", "518880.SS": "黄金ETF"}

print(f"{'日期':<25} {'方向':>4}  {'数量':>6}  {'标的':<12} {'成交价':>10}")
print("-" * 65)
for fill in result.trades:
    sym = fill.order.symbol
    print(
        f"{fill.filled_at:<25} {fill.order.side:>4}  "
        f"{fill.order.shares:>6}  {NAMES.get(sym, sym):<12} "
        f"{fill.filled_price:>10.4f}"
    )

---
## 8. 查看宽表

引擎运行后，每个 symbol 的 DataFrame 都被逐步加宽：

```
原始行情 → +mom +vol +ram → +tw
```

In [ ]:
df_nasdaq = result.mktdata["513100.SS"]
print(f"宽表列: {list(df_nasdaq.columns)}")
print()
print("纳指100ETF 最后 5 行：")
print(df_nasdaq[["close", "mom", "vol", "ram", "tw"]].tail().to_string(float_format="%.4f"))

---
## 9. 调仓频率对比

调仓频率是轮动策略的关键参数——频率太高增加交易成本，太低可能错过切换时机。

我们对比三种频率：

In [ ]:
# 公共组件
COMMON = dict(
    hypothesis="风险调整动量轮动策略",
    universe=StaticUniverse(("513100.SS", "510300.SS", "518880.SS")),
    indicators={
        "mom": (Momentum(), {"column": "close", "period": 20}),
        "vol": (RollingVolatility(), {"column": "close", "period": 20}),
        "ram": (Ratio(), {"col_a": "mom", "col_b": "vol"}),
    },
    signals={
        "tw": (TopNRanking(), {"score": "ram", "n": 2, "max_weight": 0.6}),
    },
    entry_rules=[],
    exit_rules=[],
)

variants = {
    "每5天": 5,
    "每10天": 10,
    "每20天": 20,
}

results = {}
for label, freq in variants.items():
    b = SimBroker()
    r = Engine().run(
        Strategy(
            name=f"rebal_{freq}d",
            rebalance_rules=[RebalanceRule(weight_col="tw", frequency=freq)],
            **COMMON,
        ),
        market=LocalMarketDataProvider(),
        router=b, receiver=b,
        start="2024-11-15", end="2026-02-28",
        initial_cash=100_000.0,
    )
    results[label] = r

# 对比表
header = f"{'':>16}" + "".join(f"{label:>12}" for label in results)
print(header)
print("-" * len(header))
for metric, fn in [
    ("总收益率", lambda r: f"{r.total_return():.2%}"),
    ("年化收益率", lambda r: f"{r.annualized_return():.2%}"),
    ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
    ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
    ("交易次数", lambda r: f"{len(r.trades)}"),
    ("期末资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
]:
    vals = "".join(f"{fn(r):>12}" for r in results.values())
    print(f"{metric:>16}{vals}")

---
## 10. 分阶段执行

和 SMA 均线策略一样，轮动策略也支持 `run_through` 分阶段执行。

这在调试时非常有用——先验证 Indicator 是否合理，再看 Signal 权重分配是否符合预期，最后才加入 RebalanceRule：

In [ ]:
# 只执行到 Signal 阶段 — 观察权重分配
b = SimBroker()
result_sig = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    router=b, receiver=b,
    start="2024-11-15", end="2026-02-28",
    run_through="signal",
)

print(f"交易次数: {len(result_sig.trades)}（预期为 0，Signal 阶段不执行 Rule）")
print()

# 查看某一天三只 ETF 的权重
for sym, name in NAMES.items():
    df = result_sig.mktdata[sym]
    cols = [c for c in ["close", "mom", "vol", "ram", "tw"] if c in df.columns]
    sample = df.loc["2025-06-01":"2025-06-10", cols]
    if not sample.empty:
        print(f"\n{name}:")
        print(sample.to_string(float_format="%.4f"))

---
## 小结

本教程覆盖了轮动策略的完整构建流程：

| 组件 | 职责 | 关键参数 |
|------|------|----------|
| `Momentum` | 计算 N 日动量 | period=20 |
| `RollingVolatility` | 计算 N 日波动率 | period=20 |
| `Ratio` | 两列之比（风险调整动量） | col_a, col_b |
| `TopNRanking` | 截面排名 → 目标权重 | score, n, max_weight |
| `RebalanceRule` | 按权重定期调仓 | weight_col, frequency |

### 与均线策略的对比

| | 均线交叉 | 轮动策略 |
|---|---------|----------|
| Signal 输出 | 布尔值 | 权重（0~1） |
| Rule 类型 | Entry + Exit | Rebalance |
| 仓位管理 | 信号驱动 | 周期驱动 |
| 多标的 | 独立持仓 | 截面竞争 |

### 关键设计原则

- **指标预热期** — Momentum(20) + RollingVolatility(20) 需要 ~21 个交易日，回测开始日期应提前覆盖
- **权重上限归现金** — 超过 max_weight 的部分不重新分配，自动降低总仓位
- **频率门控** — 用 frequency 参数控制调仓频率，权衡换手率与跟踪精度
- **架构不变** — 同样是 Universe → Indicator → Signal → Rule 四阶段，同样通过三接口解耦